In [2]:
import time
import requests
import pandas as pd

products_to_find = ['турник', 'пояс с цепью для отягощения', 'упоры для отжиманий', 'фитнес резинки', 'брусья', 'гимнастические кольца']

url = "https://search.wb.ru/exactmatch/ru/common/v18/search"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36',
    'Accept': '*/*',
    'Accept-Language': 'ru-RU,ru;q=0.9'
}

all_products = []
max_pages = 15

for product in products_to_find:
    for page in range(1, max_pages + 1):
        
        params = {
            'appType': '1',
            'curr': 'rub',
            'dest': '-1257786',
            'query': product,
            'resultset': 'catalog',
            'sort': 'popular',
            'spp': '30',
            'page': str(page)
        }
        
        max_retries = 5
        retry_delay = 5
        page_products = []
        
        for attempt in range(max_retries):
            response = requests.get(url, params=params, headers=headers)
            
            if response.status_code == 200:
                data = response.json()
                page_products = data.get('products', [])
                for page_product in page_products:
                    page_product['category'] = product
                break
                
            elif response.status_code == 429:
                wait_seconds = retry_delay * (attempt + 1)
                print(f"Попытка {attempt + 1}: Ошибка 429. Ждем {wait_seconds} сек")
                time.sleep(wait_seconds)
            else:
                print(f"Ошибка {response.status_code}. Пропускаем страницу")
                break
        
        if not page_products:
            print("Товары закончились или произошла ошибка. Прерываем цикл постраничного сбора")
            break
            
        all_products.extend(page_products)
        print(f"На странице {page} найдено товаров: {len(page_products)} для категории: '{product}'")
        
        time.sleep(5)
    time.sleep(10)

print(f"\nСбор завершен. Всего собрано товаров: {len(all_products)}")


Попытка 1: Ошибка 429. Ждем 5 сек
На странице 1 найдено товаров: 100 для категории: 'турник'
На странице 2 найдено товаров: 100 для категории: 'турник'
На странице 3 найдено товаров: 100 для категории: 'турник'
На странице 4 найдено товаров: 100 для категории: 'турник'
Попытка 1: Ошибка 429. Ждем 5 сек
На странице 5 найдено товаров: 100 для категории: 'турник'
На странице 6 найдено товаров: 100 для категории: 'турник'
На странице 7 найдено товаров: 100 для категории: 'турник'
На странице 8 найдено товаров: 100 для категории: 'турник'
На странице 9 найдено товаров: 100 для категории: 'турник'
На странице 10 найдено товаров: 100 для категории: 'турник'
На странице 11 найдено товаров: 100 для категории: 'турник'
На странице 12 найдено товаров: 100 для категории: 'турник'
Товары закончились или произошла ошибка. Прерываем цикл постраничного сбора
На странице 1 найдено товаров: 100 для категории: 'пояс с цепью для отягощения'
На странице 2 найдено товаров: 100 для категории: 'пояс с цепью д

In [4]:
df = pd.DataFrame(all_products)
df.to_csv('WorkoutProducts.csv')

df['category'].value_counts()

category
упоры для отжиманий            1500
фитнес резинки                 1500
брусья                         1500
гимнастические кольца          1500
турник                         1200
пояс с цепью для отягощения    1100
Name: count, dtype: int64